
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Online Evaluation with Multi-Turn Conversations

## Overview

In this lab, you are the ML engineer responsible for a production travel agent. The setup cell generates multi-turn conversation traces tagged with session IDs. Your job is to deploy multi-turn evaluation judges, verify they are running, and tune their sampling rates for production.

**This lab assumes familiarity with online evaluation architecture, scorer types, and sampling strategies. You will now apply those concepts hands-on.**

## Learning Objectives

1. Configure multi-turn judges and verify they are active using `list_scorers()`
2. Inspect and operate the trace metrics computation job
3. Create custom scorers with the `@scorer` decorator and appropriate aggregations
4. Manage the scorer lifecycle: stop, update, and delete

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Lab Structure</strong>
<p style="margin: 8px 0 0 0; color: #333;">Each section follows a <strong>build → verify</strong> pattern. TODO cells contain code you write; instructional cells guide you through the UI with expandable steps. Expand the <strong>Answer</strong> dropdown if you get stuck.</p>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Setup

In [0]:
%run ./Includes/Classroom-Setup-5

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 86.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.6
    Uninstalling protobuf-6.33.6:
      Successfully uninstalled protobuf-6.33.6
  Attempting uninstall: openai
    Found existing installation: openai 2.30.0
    Uninstalling openai-2.30.0:
      Successfully uninstalled openai-2.30.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 1.1.6 requires openai<3.0.0,>=1.109.1, but you have openai 3.39.0 which is

Starting Demo Environment Setup
Using config: Includes/config/config_5.yaml

[0] Compute check passed: Serverless 5 detected.

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17710504_1793055741' found. Using it.

[2] Setting up schema...
  Catalog : labuser17710504_1793055741
  Schema  : agent_monitoring

[3] Setting up MLflow...
  Setting tracking URI: databricks
Setting up experiment: evaluation-and-monitoring-training
Experiment exists. Deleting and recreating: evaluation-and-monitoring-training
✅ Deleted experiment 3970643669269960
✅ Created experiment: /Workspace/Users/labuser17710504_1793055741@vocareum.com/evaluation-and-monitoring-training
  Active experiment: /Workspace/Users/labuser17710504_1793055741@vocareum.com/evaluation-and-monitoring-training

[4] Generating synthetic traces...


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


  [user_101] Turn 1: I'm planning a trip to Japan in April. What should...
  [user_101] Turn 2: What about cherry blossoms — will I catch them?...
  [user_101] Turn 3: Any hotel recommendations in Kyoto?...
  [user_202] Turn 1: I need to book a flight from NYC to London. What a...
  [user_202] Turn 2: That's not very helpful. Which airlines specifical...
  [user_202] Turn 3: Okay, but what about prices? And are there direct ...
  [user_303] Turn 1: I need to cancel my hotel reservation in Barcelona...
  [user_303] Turn 2: No, I said I need to CANCEL my reservation. Confir...
  [user_303] Turn 3: I've already asked twice. I need you to cancel res...

  Logged 9 traces across 3 sessions (no LLM calls)
Refreshing environment

Demo Environment Setup Complete


[Trace(trace_id=tr-4417c11df52b4e5238917bb7b0f89314), Trace(trace_id=tr-57824ecc014027e52e3183ee2b69423e), Trace(trace_id=tr-ecf30faec0d2c7f8928d4bfe3604162f), Trace(trace_id=tr-b74b19341e9625f69c5ce745c0c5002a), Trace(trace_id=tr-21cd8efc66c0bd509d53ff4da7e37558), Trace(trace_id=tr-899abc5fc93303035bddc2ddda8815c1), Trace(trace_id=tr-aced509d55ddc70852704d6d78c4c04c), Trace(trace_id=tr-7cd687acfec0c20d566d123a54dd1b01), Trace(trace_id=tr-53908f4759773fa146d9224eedc06db5)]

## B. Multi-Turn Judges

<div style="background: #1B5162; color: white; padding: 20px 24px; border-radius: 10px; margin: 12px 0;">
<div style="font-size: 11pt; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Scenario</div>
<div style="font-size: 14pt; line-height: 1.5;">Users have reported that the agent sometimes gives incomplete answers or takes too many turns to resolve simple questions. You need multi-turn evaluation judges to detect these patterns across entire conversations.</div>
</div>

### B1. Create and Deploy Conversation Judges

The user complaints describe two distinct problems: **incomplete answers** (the agent doesn't fully resolve the request) and **too many turns** (users have to repeat themselves). Single-turn safety checks can't catch these because they evaluate each response in isolation. You need multi-turn judges that analyze the full conversation thread to detect these patterns. Each judge below targets a specific dimension of the problem:

<div style="display: flex; gap: 16px; margin: 16px 0;">
  <div style="flex: 1; background: #F8F9FC; border-radius: 10px; padding: 16px 18px; border-left: 4px solid #2574B5;">
    <div style="font-weight: 700; color: #2574B5; font-size: 12pt; margin-bottom: 6px;">ConversationCompleteness</div>
    <div style="font-size: 12pt; color: #444;">Did the agent fully resolve the request?</div>
  </div>
  <div style="flex: 1; background: #F8F9FC; border-radius: 10px; padding: 16px 18px; border-left: 4px solid #FE3722;">
    <div style="font-weight: 700; color: #FE3722; font-size: 12pt; margin-bottom: 6px;">UserFrustration</div>
    <div style="font-size: 12pt; color: #444;">Is the user showing signs of frustration?</div>
  </div>
  <div style="flex: 1; background: #F8F9FC; border-radius: 10px; padding: 16px 18px; border-left: 4px solid #02A36F;">
    <div style="font-weight: 700; color: #02A36F; font-size: 12pt; margin-bottom: 6px;">ConversationalSafety</div>
    <div style="font-size: 12pt; color: #444;">Are there safety concerns across the session?</div>
  </div>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">What happens when you call .start()</strong>
<p style="margin: 8px 0 0 0; color: #333;">Calling <code>.register()</code> and <code>.start()</code> activates <strong>online quality monitoring</strong> for your experiment. Databricks creates a background <strong>trace metrics computation job</strong> that continuously:</p>
<ul style="margin: 8px 0 0 16px; color: #333;">
<li>Pulls new traces from the experiment</li>
<li>Groups them into conversations via <code>mlflow.trace.session</code> metadata</li>
<li>Runs your scorers against each conversation</li>
<li>Logs the resulting assessments back as feedback on the traces</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;">This job is created automatically the first time you call <code>.start()</code>. It keeps running as long as you have active scorers (<code>sample_rate > 0</code>).</p>
</div>
</div>

**TODO:**
1. Import all three judges from `mlflow.genai.scorers`, register them with unique names, and start them.
    - Safety is critical: **sample_rate=1.0**
    - Completeness and frustration are expensive LLM judges: **sample_rate=0.5**

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Step 1: Import the three multi-turn judges + ScorerSamplingConfig
#         from mlflow.genai.scorers
# Step 2: Use mlflow.set_experiment() with mlflow_experiment_path
# Step 3: Instantiate, register (name= must be unique), and start each
#
# Safety:       sample_rate=1.0
# Completeness: sample_rate=0.5
# Frustration:  sample_rate=0.5
# ──────────────────────────────────────────────────────────────────────

import mlflow
mlflow.set_experiment(mlflow_experiment_path)

from mlflow.genai.scorers import (
    ConversationCompleteness,
    UserFrustration,
    ConversationalSafety,
    ScorerSamplingConfig,
)

completeness = ConversationCompleteness().register(name="conversation_completeness")
completeness = completeness.start(
    sampling_config=ScorerSamplingConfig(sample_rate=0.5),
)

frustration = UserFrustration().register(name="user_frustration")
frustration = frustration.start(
    sampling_config=ScorerSamplingConfig(sample_rate=0.5),
)

safety = ConversationalSafety().register(name="conversational_safety")
safety = safety.start(
    sampling_config=ScorerSamplingConfig(sample_rate=1.0),
)


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> B1. Create and Deploy Conversation Judges
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import mlflow
mlflow.set_experiment(mlflow_experiment_path)

from mlflow.genai.scorers import (
    ConversationCompleteness,
    UserFrustration,
    ConversationalSafety,
    ScorerSamplingConfig,
)

completeness = ConversationCompleteness().register(name="conversation_completeness")
completeness = completeness.start(
    sampling_config=ScorerSamplingConfig(sample_rate=0.5),
)

frustration = UserFrustration().register(name="user_frustration")
frustration = frustration.start(
    sampling_config=ScorerSamplingConfig(sample_rate=0.5),
)

safety = ConversationalSafety().register(name="conversational_safety")
safety = safety.start(
    sampling_config=ScorerSamplingConfig(sample_rate=1.0),
)
```

</div>
</details>

### B2. Add a Custom Judge via the UI

Before running the job, add one more judge. this time through the UI instead of code. This demonstrates that you can mix programmatic and UI-created judges on the same experiment. The job will evaluate all active judges in a single run.

Click each step below to expand the instructions.

<br></br>

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<style>
.lab-acc-item {
  border: 2px solid #e0e0e0;
  border-radius: 8px;
  margin-bottom: 8px;
  overflow: hidden;
  transition: border-color 0.2s;
}
.lab-acc-item.open { border-color: #2574B5; }
.lab-acc-header {
  display: flex;
  align-items: center;
  gap: 14px;
  padding: 14px 18px;
  cursor: pointer;
  user-select: none;
  background: #F9F7F4;
  transition: background 0.15s;
}
.lab-acc-item.open .lab-acc-header { background: #E8F1FA; }
.lab-acc-num {
  width: 32px;
  height: 32px;
  border-radius: 50%;
  background: #e0e0e0;
  color: #666;
  font-size: 13pt;
  font-weight: 700;
  display: flex;
  align-items: center;
  justify-content: center;
  flex-shrink: 0;
  transition: background 0.2s, color 0.2s;
}
.lab-acc-item.open .lab-acc-num { background: #2574B5; color: #fff; }
.lab-acc-summary {
  flex: 1;
  font-size: 13.5pt;
  font-weight: 600;
  color: #0b2026;
}
.lab-acc-chevron {
  font-size: 14pt;
  color: #888;
  transition: transform 0.25s;
}
.lab-acc-item.open .lab-acc-chevron { transform: rotate(180deg); color: #2574B5; }
.lab-acc-body {
  max-height: 0;
  opacity: 0;
  overflow: hidden;
  transition: max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease;
  padding: 0 18px 0 64px;
  font-size: 14pt;
  line-height: 1.7;
  color: #333;
}
.lab-acc-item.open .lab-acc-body {
  max-height: 600px;
  opacity: 1;
  padding: 4px 18px 18px 64px;
}
</style>

<div id="lab-acc-b2"></div>

</div>

<script>
var LAB_B2 = [
  {
    summary: 'Navigate to the experiment',
    detail: 'Navigate to <strong>Experiments</strong> in the left menu. Search for and click on <strong>evaluation-and-monitoring-training</strong>.'
  },
  {
    summary: 'Open the Scorers panel',
    detail: 'In your experiment, click <strong>Scorers</strong> under <strong>Evaluation</strong>, then click <strong>New LLM scorer</strong>.'
  },
  {
    summary: 'Configure the scorer',
    detail: 'Click <strong>Sessions</strong> under <strong>General</strong>. Name it <strong>professional</strong>. Under <strong>Evaluation criteria</strong>, select <strong>Custom scorer</strong> and enter: <em>Evaluate if the interaction is professional {{conversation}}</em>'
  },
  {
    summary: 'Create',
    detail: 'Click <strong>Create Scorers</strong> at the bottom right. This will populate your experiment with a new scorer called <strong>professional</strong>.'
  }
];

(function() {
  var container = document.getElementById('lab-acc-b2');
  LAB_B2.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'lab-acc-item' + (i === 0 ? ' open' : '');
    div.innerHTML =
      '<div class="lab-acc-header" onclick="labToggleB2(' + i + ')">'
      + '<div class="lab-acc-num">' + (i + 1) + '</div>'
      + '<div class="lab-acc-summary">' + item.summary + '</div>'
      + '<div class="lab-acc-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="lab-acc-body">' + item.detail + '</div>';
    container.appendChild(div);
  });
})();

function labToggleB2(idx) {
  var items = document.querySelectorAll('#lab-acc-b2 .lab-acc-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

### B3. Add a Custom Scorer in Code

The built-in judges evaluate quality dimensions like safety and completeness, but your team also wants to track a basic metric: **response length**. Overly long responses slow down users and increase token costs; overly short ones may indicate the agent is not being helpful. A custom scorer lets you measure this programmatically and get mean/min/max aggregations across all evaluated traces.

This demonstrates three ways to deploy scorers: out-of-the-box (B1), via the UI (B2), and custom code (B3). All will run together when the job executes.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Serialization Rules</strong>
<p style="margin: 8px 0 0 0; color: #333;">Custom scorers run remotely. All <code>import</code> statements must be <strong>inside the function body</strong>. Return consistent types. Handle missing data gracefully.</p>
</div>
</div>

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Create a @scorer function that:
#   - Measures len(str(outputs.get("response", "")))
#   - Has aggregations=["mean", "min", "max"]
# Then register as "response_length" and start with sample_rate=0.5
#
# Hint: from mlflow.genai.scorers import scorer, ScorerSamplingConfig
# ──────────────────────────────────────────────────────────────────────

from mlflow.genai.scorers import scorer, ScorerSamplingConfig

@scorer(aggregations=["mean", "min", "max"])
def response_length(outputs):
    """Measure response length in characters"""
    return len(str(outputs.get("response", "")))

length_scorer = response_length.register(name="response_length")
length_scorer = length_scorer.start(
    sampling_config=ScorerSamplingConfig(sample_rate=0.5)
)



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> B3. Custom Scorer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.genai.scorers import scorer, ScorerSamplingConfig

@scorer(aggregations=["mean", "min", "max"])
def response_length(outputs):
    """Measure response length in characters"""
    return len(str(outputs.get("response", "")))

length_scorer = response_length.register(name="response_length")
length_scorer = length_scorer.start(
    sampling_config=ScorerSamplingConfig(sample_rate=0.5)
)
```

</div>
</details>

### B4. Verify Sessions and Run the Job

All five scorers are now deployed (3 out-of-the-box + 1 UI-created + 1 custom code). Verify the traces exist and run the job once to evaluate everything together. Click each step to expand.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Default Conversation Timing</strong>
<p style="margin: 8px 0 0 0; color: #333;">Multi-turn judges run after a conversation is considered complete. By default, no new traces with that session ID for <strong>5 minutes</strong>. Make sure the setup script was ran at least 5 minutes ago.</p>
</div>
</div>

<br></br>

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<style>
.lab-acc-item {
  border: 2px solid #e0e0e0;
  border-radius: 8px;
  margin-bottom: 8px;
  overflow: hidden;
  transition: border-color 0.2s;
}
.lab-acc-item.open { border-color: #2574B5; }
.lab-acc-header {
  display: flex;
  align-items: center;
  gap: 14px;
  padding: 14px 18px;
  cursor: pointer;
  user-select: none;
  background: #F9F7F4;
  transition: background 0.15s;
}
.lab-acc-item.open .lab-acc-header { background: #E8F1FA; }
.lab-acc-num {
  width: 32px;
  height: 32px;
  border-radius: 50%;
  background: #e0e0e0;
  color: #666;
  font-size: 13pt;
  font-weight: 700;
  display: flex;
  align-items: center;
  justify-content: center;
  flex-shrink: 0;
  transition: background 0.2s, color 0.2s;
}
.lab-acc-item.open .lab-acc-num { background: #2574B5; color: #fff; }
.lab-acc-summary {
  flex: 1;
  font-size: 13.5pt;
  font-weight: 600;
  color: #0b2026;
}
.lab-acc-chevron {
  font-size: 14pt;
  color: #888;
  transition: transform 0.25s;
}
.lab-acc-item.open .lab-acc-chevron { transform: rotate(180deg); color: #2574B5; }
.lab-acc-body {
  max-height: 0;
  opacity: 0;
  overflow: hidden;
  transition: max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease;
  padding: 0 18px 0 64px;
  font-size: 14pt;
  line-height: 1.7;
  color: #333;
}
.lab-acc-item.open .lab-acc-body {
  max-height: 600px;
  opacity: 1;
  padding: 4px 18px 18px 64px;
}
</style>

<div id="lab-acc-b4"></div>

</div>

<script>
var LAB_B4 = [
  {
    summary: 'Verify sessions in the Experiment UI',
    detail: 'Navigate to <strong>Experiments</strong> in the left menu. Search for and click on <strong>evaluation-and-monitoring-training</strong>. Click on <strong>Sessions</strong> and note that your sessions have been populated. Click the dropdown next to a session to view its 3 turns.'
  },
  {
    summary: 'Find the Trace Metrics Computation Job',
    detail: 'Navigate to <strong>Jobs &amp; Pipelines</strong> and search for <em>Trace Metrics Computation Job</em>. Your job has the experiment ID prepended, e.g., <strong>[0123456789] Trace Metrics Computation Job</strong>. It runs on a fixed ~15-minute schedule. Your <code>sample_rate</code> controls how many traces are evaluated, not how often the job runs.'
  },
  {
    summary: 'Pause the schedule and run manually',
    detail: 'For this lab, click <strong>Pause</strong> to stop the automatic schedule. Then click <strong>Run now</strong>. This single run will evaluate traces using all five active judges: the three multi-turn programmatic judges, the <strong>professional</strong> judge you created via the UI, and the <strong>response_length</strong> custom scorer.'
  },
  {
    summary: 'Inspect the job output with Genie Code',
    detail: 'After the job completes, navigate to <strong>Runs</strong> and click the last successful run (green bar). Review the log output. Try <strong>Genie Code</strong> (top right next to your workspace name) and ask: <em>"What does this output tell me?"</em> Genie Code should respond with something like: <em>"In short, this output confirms that the agent monitoring metrics computation completed without errors for this run."</em> Read through the full response to understand what was evaluated.'
  },
  {
    summary: 'View scorer results on traces',
    detail: 'Navigate back to the experiment. Click on <strong>Sessions</strong> to see session-level feedback. note that <strong>UserFrustration</strong> and <strong>ConversationCompleteness</strong> results appear only at the session level (on the first trace of each session), not on individual traces. Click <strong>Traces</strong> to see per-trace assessments from <strong>Safety</strong>, <strong>professional</strong>, and <strong>response_length</strong>.<br><br><strong>Take a moment to read the judge feedback carefully.</strong> Do you agree with the assessments? For example, does the completeness judge correctly flag Session 2 (vague flight answers) as incomplete? Does the frustration judge catch Session 3 (ignored cancellation request)? Evaluating whether you agree with the judges is a critical step in tuning your evaluation strategy.'
  }
];

(function() {
  var container = document.getElementById('lab-acc-b4');
  LAB_B4.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'lab-acc-item' + (i === 0 ? ' open' : '');
    div.innerHTML =
      '<div class="lab-acc-header" onclick="labToggleB4(' + i + ')">'
      + '<div class="lab-acc-num">' + (i + 1) + '</div>'
      + '<div class="lab-acc-summary">' + item.summary + '</div>'
      + '<div class="lab-acc-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="lab-acc-body">' + item.detail + '</div>';
    container.appendChild(div);
  });
})();

function labToggleB4(idx) {
  var items = document.querySelectorAll('#lab-acc-b4 .lab-acc-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

### B5. Verify Your Scorers Are Running

Confirm all five scorers were deployed correctly and are active. In production, this is the same audit step you would run to verify what is evaluating your traces and at what rates.

**TODO**: Use `list_scorers()` to enumerate all registered scorers. You should see 5 active scorers.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Import list_scorers from mlflow.genai.scorers
# List all scorers and print their name and sample rate
#
# Hint: scorer objects have .name and .sample_rate
# ──────────────────────────────────────────────────────────────────────
from mlflow.genai.scorers import list_scorers

scorers = list_scorers()
print(f"Found {len(scorers)} registered scorers:\n")
for s in scorers:
    status = "ACTIVE" if s.sample_rate > 0 else "STOPPED"
    print(f"  {s.name}: rate={s.sample_rate} [{status}]")


Found 5 registered scorers:

  conversation_completeness: rate=0.5 [ACTIVE]
  user_frustration: rate=0.5 [ACTIVE]
  conversational_safety: rate=1.0 [ACTIVE]
  response_length: rate=0.5 [ACTIVE]
  Professional: rate=1.0 [ACTIVE]



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> B5. Verify Scorers
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.genai.scorers import list_scorers

scorers = list_scorers()
print(f"Found {len(scorers)} registered scorers:\n")
for s in scorers:
    status = "ACTIVE" if s.sample_rate > 0 else "STOPPED"
    print(f"  {s.name}: rate={s.sample_rate} [{status}]")
```

</div>
</details>

### B6. Stop Deployed Scorers
You have confirmed the scorers work and seen their assessments. Before moving to Section C, stop the current scorers. In production, you would stop a scorer when you want to pause evaluation temporarily without losing the scorer's registration. The background job stops running once all scorers have `sample_rate = 0`.

**TODO**: Stop all five scorers: `completeness`, `frustration`, `safety`, `length_scorer` (from B3), and the `professional` judge (use `get_scorer(name="professional")` since it was created via the UI). Note that `.stop()` returns a new Python object . Always reassign (e.g., `safety = safety.stop()`) so your local variable reflects the updated state.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Stop all 5 scorers:
# - completeness, frustration, safety (local variables from B1)
# - length_scorer (local variable from B3)
# - professional (use get_scorer(name="professional") to fetch it first)
# Note: .stop() returns a new Python object. reassign so your variable reflects the change
# ──────────────────────────────────────────────────────────────────────
from mlflow.genai.scorers import get_scorer

# Programmatic scorers (B1)
completeness = completeness.stop()
print(f"Sample rate for completeness: {completeness.sample_rate}")
frustration = frustration.stop()
print(f"Sample rate for frustration: {frustration.sample_rate}")
safety = safety.stop()
print(f"Sample rate for safety: {safety.sample_rate}")

# Custom scorer (B3)
length_scorer = length_scorer.stop()
print(f"Sample rate for response_length: {length_scorer.sample_rate}")

# UI-created scorer (B2). fetch with get_scorer since no local variable
professional = get_scorer(name="Professional")
professional = professional.stop()
print(f"Sample rate for professional: {professional.sample_rate}")


Sample rate for completeness: 0.0
Sample rate for frustration: 0.0
Sample rate for safety: 0.0
Sample rate for response_length: 0.0
Sample rate for professional: 0.0



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> B6. Stop Deployed Scorers
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.genai.scorers import get_scorer

# Programmatic scorers (B1)
completeness = completeness.stop()
print(f"Sample rate for completeness: {completeness.sample_rate}")
frustration = frustration.stop()
print(f"Sample rate for frustration: {frustration.sample_rate}")
safety = safety.stop()
print(f"Sample rate for safety: {safety.sample_rate}")

# Custom scorer (B3)
length_scorer = length_scorer.stop()
print(f"Sample rate for response_length: {length_scorer.sample_rate}")

# UI-created scorer (B2). fetch with get_scorer since no local variable
professional = get_scorer(name="professional")
professional = professional.stop()
print(f"Sample rate for professional: {professional.sample_rate}")
```

</div>
</details>

## C. Sampling Strategy

<div style="background: #1B5162; color: white; padding: 20px 24px; border-radius: 10px; margin: 12px 0;">
<div style="font-size: 11pt; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Scenario</div>
<div style="font-size: 14pt; line-height: 1.5;">Your scorers have been running and the quality signals are stabilizing. Now you need to tune the sampling rates to balance cost and coverage for production.</div>
</div>

### C1. Update a Sampling Rate

After running the judges for a while, you notice the completeness scorer's results have stabilized. the pass rate hasn't changed in the last several evaluation cycles. Continuing to evaluate 50% of traces is wasting LLM calls. You can reduce the rate to 5% to save costs while still catching regressions. This is the typical production lifecycle: start high for fast feedback, dial back once the signal is stable.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Return value pattern</strong>
<p style="margin: 8px 0 0 0; color: #333;"><code>.update()</code> changes the scorer's configuration on the server but returns a <strong>new Python object</strong> reflecting the change. Always reassign: <code>scorer = scorer.update(...)</code></p>
</div>
</div>

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# 1. Use get_scorer() to fetch "conversation_completeness"
# 2. Update its sample_rate to 0.05
# 3. Print the original and updated rates to show the return value pattern
# 4. List all scorers to see the full picture
#
# Hint: from mlflow.genai.scorers import get_scorer, ScorerSamplingConfig
# ──────────────────────────────────────────────────────────────────────
from mlflow.genai.scorers import get_scorer, list_scorers, ScorerSamplingConfig

# Fetch and update
original = get_scorer(name="conversation_completeness")
print(f"Original rate: {original.sample_rate}")

updated = original.update(
    sampling_config=ScorerSamplingConfig(sample_rate=0.05)
)
print(f"Updated rate:  {updated.sample_rate}")

# Full audit
print("\nAll scorers:")
for s in list_scorers():
    status = "ACTIVE" if s.sample_rate > 0 else "STOPPED"
    print(f"  {s.name}: rate={s.sample_rate} [{status}]")


Original rate: 0.0
Updated rate:  0.05

All scorers:
  conversation_completeness: rate=0.05 [ACTIVE]
  user_frustration: rate=0.0 [STOPPED]
  conversational_safety: rate=0.0 [STOPPED]
  response_length: rate=0.0 [STOPPED]
  Professional: rate=0.0 [STOPPED]



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> C1. Update and Verify
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.genai.scorers import get_scorer, list_scorers, ScorerSamplingConfig

# Fetch and update
original = get_scorer(name="conversation_completeness")
print(f"Original rate: {original.sample_rate}")

updated = original.update(
    sampling_config=ScorerSamplingConfig(sample_rate=0.05)
)
print(f"Updated rate:  {updated.sample_rate}")

# Full audit
print("\nAll scorers:")
for s in list_scorers():
    status = "ACTIVE" if s.sample_rate > 0 else "STOPPED"
    print(f"  {s.name}: rate={s.sample_rate} [{status}]")
```

</div>
</details>

## D. Clean Up Scorers
In a shared workspace, abandoned scorers continue consuming resources. The background job keeps running, making LLM calls on every evaluation cycle. Deleting scorers you no longer need is important for cost management. Unlike `.stop()` (which pauses but keeps the scorer registered), `delete_scorer()` permanently removes it. Historical assessments already written to traces are preserved.

**TODO**: List all scorers, delete each one using `delete_scorer(name=...)`, and verify the list is empty.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# 1. List all scorers using list_scorers()
# 2. Delete each one using delete_scorer(name=...)
# 3. Verify the list is empty
#
# Hint: from mlflow.genai.scorers import list_scorers, delete_scorer
# ──────────────────────────────────────────────────────────────────────
from mlflow.genai.scorers import list_scorers, delete_scorer

scorers = list_scorers()
print(f"Deleting {len(scorers)} scorers:\n")

for s in scorers:
    print(f"  Deleting: {s.name}...", end=" ")
    delete_scorer(name=s.name)
    print("done")

# Verify
remaining = list_scorers()
print(f"\nRemaining scorers: {len(remaining)}")


Deleting 5 scorers:

  Deleting: conversation_completeness... done
  Deleting: user_frustration... done
  Deleting: conversational_safety... done
  Deleting: response_length... done
  Deleting: Professional... done

Remaining scorers: 0



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> D. Clean Up Scorers
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.genai.scorers import list_scorers, delete_scorer

scorers = list_scorers()
print(f"Deleting {len(scorers)} scorers:\n")

for s in scorers:
    print(f"  Deleting: {s.name}...", end=" ")
    delete_scorer(name=s.name)
    print("done")

# Verify
remaining = list_scorers()
print(f"\nRemaining scorers: {len(remaining)}")
```

</div>
</details>

## Conclusion

1. **Multi-turn judges** evaluate entire conversations using the `mlflow.trace.session` metadata to group traces into sessions
2. **Custom scorers** use the `@scorer` decorator with all imports inline and support `aggregations` for numeric metrics
3. **Sampling rates** should be tuned by scorer type: 100% for safety, 5% for expensive LLM judges once stable, 30-50% during active development
4. **Return value pattern**: `.update()` and `.stop()` change the server state but return new Python objects -- always reassign so your local variable reflects the change
5. **`list_scorers()`** lets you audit what is running and at what rates

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
